In [42]:
# xarray to read NETCDF
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import dask as dd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'darkgrey'

In [43]:
sst = xr.open_dataset('data/SST/sst.mnmean.nc')
sst = sst.assign_coords(lon=(((sst.lon + 180) % 360) - 180)).sortby(['lon'])
sst_df = sst.sel(time=slice('1993-01-01', '2024-12-31')).to_dataframe().reset_index()
sst_df = sst_df.query('time_bnds > 0 and nbnds == 0')
sst_df['month'] = sst_df['time'].dt.month
sst_df['year'] = sst_df['time'].dt.year

In [44]:
chirps = xr.open_dataset('data/CHIRPS/chirps-v2.0.monthly.nc')

In [45]:
ltm_sst = sst_df.dropna().groupby(['lat', 'lon', 'month']).mean('sst').reset_index()
ltm_sst['std'] = sst_df.dropna().groupby(['lat', 'lon', 'month']).std().reset_index()['sst']
ltm_sst_clean = ltm_sst.drop(['time_bnds', 'nbnds', 'year'], axis=1)
ltm_sst_clean = ltm_sst_clean.rename(columns={'sst':'ltm_sst'})

In [46]:
sst_anomaly = sst_df.drop(['time_bnds', 'nbnds'], axis=1).merge(ltm_sst_clean, on=['lat', 'lon', 'month'], how='left')
sst_anomaly['sst_anomaly'] = sst_anomaly['sst'] - sst_anomaly['ltm_sst']
sst_anomaly['normalized_sst_anomaly'] = sst_anomaly['sst_anomaly'] / sst_anomaly['std']

In [47]:
chirps_eastern_east_africa = chirps.sel(time=slice('1993-01-01', '2024-01-01'), latitude=slice(-3.5, 8), longitude=slice(38, 50)).to_dataframe().reset_index()
chirps_eastern_east_africa['month'] = chirps_eastern_east_africa['time'].dt.month
chirps_eastern_east_africa['year'] = chirps_eastern_east_africa['time'].dt.year

month_to_season = {
    3: 'MAM', 4: 'MAM', 5: 'MAM',   # March, April, May
}

season = ['MAM']

chirps_eastern_east_africa['season'] = chirps_eastern_east_africa['month'].map(month_to_season)

chirps_eastern_east_africa_season = chirps_eastern_east_africa.dropna(subset=['season']).groupby(['year', 'season'])[['precip']].mean().reset_index()

In [48]:
# Get tercile values
tercile_list = chirps_eastern_east_africa_season.quantile([0.33, 0.66], numeric_only=True)['precip'].to_list()

season_bn = chirps_eastern_east_africa_season.query(f'precip <= {tercile_list[0]}')['year'].to_list()
season_n = chirps_eastern_east_africa_season.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')['year'].to_list()
season_an = chirps_eastern_east_africa_season.query(f'{tercile_list[1]} <= precip')['year'].to_list()
season_dict = {'an': season_an, 'bn': season_bn, 'n': season_n}


year_to_category_map = {}
for category, year_list in season_dict.items():
    for year in year_list:
        year_to_category_map[year] = category

chirps_eastern_east_africa_season['tercile'] = chirps_eastern_east_africa_season['year'].map(year_to_category_map)

In [49]:
season_dict

{'an': [1995, 1997, 1998, 2003, 2006, 2010, 2013, 2015, 2018, 2020, 2023],
 'bn': [1999, 2000, 2001, 2004, 2008, 2009, 2011, 2017, 2019, 2022],
 'n': [1993, 1994, 1996, 2002, 2005, 2007, 2012, 2014, 2016, 2021]}

In [50]:
chirps_eastern_east_africa_season

,year,season,precip,tercile
0,1993,MAM,63.102024,n
1,1994,MAM,70.791168,n
2,1995,MAM,75.360275,an
3,1996,MAM,66.239967,n
4,1997,MAM,81.642693,an
5,1998,MAM,73.752174,an
6,1999,MAM,49.764690,bn
7,2000,MAM,48.286407,bn
8,2001,MAM,55.360474,bn
9,2002,MAM,66.756470,n


In [95]:
nino_34 = sst_anomaly.query('-5 <= lat <= 5 and -170<= lon <= -120').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().rename({'normalized_sst_anomaly': 'nino_34'}, axis=1)

In [96]:
nino_4 = sst_anomaly.query('-5 <= lat <= 5 and lon <= -150 or -5 <= lat <= 5 and lon >= 160').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'nino_4'}, axis=1)

In [97]:
western_west_v = sst_anomaly.query('-15 <= lat <= 20 and 120 <= lon <= 160').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'western_west_v'}, axis=1)

In [98]:
northern_west_v = sst_anomaly.query('20 <= lat <= 35 and lon <= -150 or 20 <= lat <= 35 and lon >= 160').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'northern_west_v'}, axis=1)

In [99]:
southern_west_v = sst_anomaly.query('-30 <= lat <= -15 and lon <= -150 or -30 <= lat <= -15 and lon >= 155').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'southern_west_v'}, axis=1)

In [100]:
SWIO = sst_anomaly.query('-50 <= lat <= -20 and 50 <= lon <= 70').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'SWIO'}, axis=1)

In [101]:
IOD_west = sst_anomaly.query('-10 <= lat <= 10 and 50 <= lon <= 70').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'IOD_west'}, axis=1)

In [102]:
IOD_east = sst_anomaly.query('-10 <= lat <= 0 and 90 <= lon <= 110').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'IOD_east'}, axis=1)

In [103]:
pd.concat([nino_34, nino_4, western_west_v, northern_west_v, southern_west_v, SWIO, IOD_west, IOD_east], axis=1)

,year,month,nino_34,nino_4,western_west_v,northern_west_v,southern_west_v,SWIO,IOD_west,IOD_east
0,1993,1,0.226764,0.174881,-1.899133,-1.459909,-1.509524,0.208716,-0.897428,-0.956619
1,1993,2,0.507616,0.085507,-2.310108,-1.346245,-1.592366,-0.300487,-0.987301,-0.707205
2,1993,3,0.588001,0.029334,-1.840947,-1.201143,-1.942652,-0.430448,-1.132885,-0.182749
3,1993,4,1.058289,0.016315,-1.995134,-1.057799,-1.995876,-0.099328,-1.048087,-0.742232
4,1993,5,1.422208,0.048840,-2.049948,-0.611045,-1.873041,0.738346,-0.120053,-0.733560
...,...,...,...,...,...,...,...,...,...,...
379,2024,8,-0.071970,0.717162,1.198190,0.572516,0.461606,0.864684,1.452369,0.798205
380,2024,9,-0.244867,0.365159,1.147252,0.645820,0.646922,0.943655,1.508885,1.007701
381,2024,10,-0.214796,0.308932,1.254107,1.145085,0.410736,0.645145,1.018261,1.544542
382,2024,11,-0.173877,0.296756,1.471140,1.604624,0.275565,1.684687,0.914358,2.035274
